# Exercise 1 · Check every reply against a schema

You will tighten two rules so that every bad reply is rejected.

**Goal:** Tighten two schema rules so that every bad reply is rejected.

**Why it matters:** A bad reply that gets saved becomes bad data.

**From the slides:** Slide 4, "A schema guarantees shape, not content; it can cost accuracy"; Slide 5, "Every field is required, and only the amount may be null"; Slide 7, "Validate every reply in your own code anyway".

**Before you start:** You do not write any code. To run a cell, click it and press Shift+Enter. A *schema* is the list of rules for a reply's shape. A *list* is items in square brackets: `["a", "b"]`.

**What the check measures:** your two edits, and that only replies 1 and 4 are saved.

**Time:** about 15 minutes.

**Done when:** the last cell prints `Done:`.

## The situation

You order dinner on a food delivery app. When something goes wrong, you send the app a message, such as "My pizza came an hour late." The app has an AI model turn each message into JSON (text with named fields): `order_id`, `issue`, `refund_amount`, `urgent`. A teammate wrote the rules; two are too loose.

## Step 1 · Check with the teammate's rules

Run the given cell below: 6 sample replies (simulated), the teammate's rules and a checker. The checker saves a reply that follows the rules and rejects the rest. Each line shows OK, or PROBLEM when the result is wrong.

Expect two PROBLEM lines: reply 2 (its amount is text) and reply 3 (it has no urgent) were saved. The last line is `PROBLEM: 2 bad replies were saved.` Steps 2 and 3 fix this.

In [ ]:
# Given: 6 sample replies (simulated), the teammate's rules, and the checker
import json

import jsonschema

REPLIES = {
    "reply 1": '{"order_id": "A1042", "issue": "late", "refund_amount": null, "urgent": false}',
    "reply 2": '{"order_id": "A3307", "issue": "wrong_item", "refund_amount": "about 20 dollars", "urgent": false}',
    "reply 3": '{"order_id": "A1185", "issue": "late", "refund_amount": null}',
    "reply 4": '{"order_id": "A2210", "issue": "damaged", "refund_amount": 35.5, "urgent": true}',
    "reply 5": '{"order_id": "A2093", "issue": "broken", "refund_amount": 12, "urgent": true}',
    "reply 6": "Sorry, I can't help with that request.",
}
REQUIRED = ["order_id", "issue", "refund_amount"]  # the teammate's rules
AMOUNT_TYPES = ["number", "string", "null"]


def verdict(text):
    # Save or reject one reply; never change it
    schema = {
        "required": REQUIRED,
        "properties": {
            "issue": {"enum": ["late", "damaged", "wrong_item", "other"]},
            "refund_amount": {"type": AMOUNT_TYPES},
            "urgent": {"type": "boolean"},
        },
    }
    try:
        jsonschema.validate(json.loads(text), schema)
        return "saved"
    except json.JSONDecodeError:
        return "rejected: it is not JSON"
    except jsonschema.ValidationError as error:
        return "rejected: " + error.message.replace("is a required property", "is missing")
    except Exception:
        return "cannot check: Step 2 or 3 has a typo"


BAD = {"reply 2": "its amount is text", "reply 3": "it has no urgent",
       "reply 5": "its issue is not on the list", "reply 6": "it is not JSON"}


def mark(name):
    # OK when a good reply is saved and a bad one rejected; PROBLEM otherwise
    saved = verdict(REPLIES[name]) == "saved"
    if name not in BAD:
        return "OK: a good reply" if saved else "PROBLEM: a good reply was rejected"
    return f"PROBLEM: {BAD[name]}, but it was saved" if saved else f"OK: {BAD[name]}"


def show_one(name):
    print(f"{name}: {verdict(REPLIES[name])}")
    print(mark(name) + (". Edit the TODO line, then run this cell again." if mark(name).startswith("PROBLEM") else ", so it was rejected."))


def check_all():
    for name in REPLIES:
        print(f"{name}: {verdict(REPLIES[name]).split(':')[0]:<9} {mark(name)}")
    bad_saved = sum(name in BAD and verdict(REPLIES[name]) == "saved" for name in REPLIES)
    good_lost = sum(name not in BAD and verdict(REPLIES[name]) != "saved" for name in REPLIES)
    if bad_saved:
        print(f"PROBLEM: {bad_saved} bad {'reply was' if bad_saved == 1 else 'replies were'} saved.")
    elif good_lost:
        print(f"PROBLEM: {good_lost} good {'reply was' if good_lost == 1 else 'replies were'} rejected.")
    else:
        print("OK: only the 2 good replies were saved; every bad reply was rejected.")


check_all()

## Step 2 · Require the urgent field

Every field must be there (Slide 5), but the teammate's list leaves out `urgent`. On the `# TODO` line, add a comma and `"urgent"` after `"refund_amount"`, so that the line reads:

`REQUIRED = ["order_id", "issue", "refund_amount", "urgent"]`

Then run the cell. Before your edit, the last line starts with `PROBLEM: it has no urgent, but it was saved.` After your edit, expect `OK: it has no urgent, so it was rejected.`

If you see a red error, a quote or a comma is missing: compare your line with the line above.

In [ ]:
# Step 2: the fields every reply must have
REQUIRED = ["order_id", "issue", "refund_amount"]  # TODO: add "urgent" to this list
show_one("reply 3")

## Step 3 · Stop text amounts

`"string"` means text. The app cannot pay back "about 20 dollars": it needs a number or null (Slide 5). On the `# TODO` line, delete `"string"` and its comma, so that the line reads:

`AMOUNT_TYPES = ["number", "null"]`

Then run the cell. Before your edit, the last line starts with `PROBLEM: its amount is text, but it was saved.` After your edit, expect `OK: its amount is text, so it was rejected.`

If you see a red error, a quote or a comma is missing: compare your line with the line above.

In [ ]:
# Step 3: the kinds of value refund_amount may hold
AMOUNT_TYPES = ["number", "string", "null"]  # TODO: delete "string" from this list
show_one("reply 2")

## Step 4 · Check every reply again

Run the cell. Expect OK on every line, and the last line `OK: only the 2 good replies were saved; every bad reply was rejected.`

In [ ]:
# Step 4: check every reply again, with your rules
check_all()

## Check your work

Run the cell. If it prints `Not yet:`, redo the step it names.

In [ ]:
# Check your work: both rules, then every reply
try:
    if sorted(REQUIRED) != ["issue", "order_id", "refund_amount", "urgent"]:
        found = 'REQUIRED should list the four fields. Redo Step 2 and run its cell.'
    elif sorted(AMOUNT_TYPES) != ["null", "number"]:
        found = 'AMOUNT_TYPES should be ["number", "null"]. Redo Step 3 and run its cell.'
    elif [verdict(t) == "saved" for t in REPLIES.values()] != [True, False, False, True, False, False]:
        found = "the Step 1 cell was changed. Undo that and run every cell."
    else:
        found = ""
except Exception:
    found = "a cell above has not run. Run every cell from the top."
print(f"Not yet: {found}" if found else "Done: only replies 1 and 4 were saved; the other four were rejected, none fixed.")

## What you learned

- A schema checks only the rules you write; a loose rule lets bad replies through (Slides 4 and 5).
- Check every reply in your own code; a bad one is rejected, never fixed (Slide 7).